In [1]:
import spacy
import re
import requests

nlp = spacy.load("en_core_web_sm")

In [18]:
text = """Amazon was founded by Jeff Bezos in 1994. The company is headquartered in Seattle, Washington."""

In [19]:
def build_prompt(text):
    return f"""
You are a high-precision factual claim extraction system.

Extract ONLY factual claims explicitly stated in the text.

STRICT RULES:

1. Only extract facts directly present in the text
2. DO NOT infer or add new information
3. DO NOT use placeholders like "not specified", "unknown"
4. Each claim must contain ONLY ONE fact
5. Replace pronouns with real entity names
6. Do NOT use vague subjects like "the company", "the bill"
7. Each claim must follow: SUBJECT + RELATION + OBJECT
8. Skip unclear or ambiguous statements
9. Do NOT include opinions or general statements
10.Always include full entity type when needed (e.g., "the film King",, "an interview featuring Yaseen Khan","actor sharukh")
11.Avoid generic phrasing like "is misleading"; instead use "stated that" or "reported that"

OUTPUT:
- Only numbered claims
- No explanations
- No extra text

TEXT:
{text}
"""

In [20]:
def llm_extract_claims(text):

    prompt = build_prompt(text)

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3",
                "prompt": prompt,
                "stream": False
            },
            timeout=60
        )

        if response.status_code != 200:
            print("❌ API Error:", response.text)
            return []

        data = response.json()

        output = data.get("response") or data.get("message", {}).get("content", "")

        if not output:
            print("⚠️ Empty response:", data)
            return []

        claims = []
        for line in output.split("\n"):
            line = line.strip()

            if not line:
                continue

            # remove numbering
            line = re.sub(r'^\d+[\).\-\s]*', '', line)

            claims.append(line)

        return claims

    except Exception as e:
        print("❌ Error:", str(e))
        return []

In [21]:
def clean_claims(claims):

    cleaned = []

    for c in claims:
        c_lower = c.lower()

        # remove LLM meta text
        if any(x in c_lower for x in [
            "here is", "here are", "note:", "output", "format"
        ]):
            continue

        # remove short / weak claims
        if len(c.split()) < 5:
            continue

        cleaned.append(c.strip())

    return list(set(cleaned))

In [22]:
def remove_hallucinations(claims):

    valid = []

    for c in claims:
        c_lower = c.lower()

        if any(x in c_lower for x in [
            "not specified", "unknown", "unclear"
        ]):
            continue

        valid.append(c)

    return valid

In [23]:
def strict_validation(claims, text):

    validated = []
    text_lower = text.lower()

    for c in claims:

        words = c.lower().split()

        overlap = sum(1 for w in words if w in text_lower)

        if overlap < 4:
            continue

        if c.lower().startswith(("this", "these", "those")):
            continue

        validated.append(c)

    return list(set(validated))

In [24]:
def score_claim(claim):

    doc = nlp(claim)

    entity_score = len(doc.ents)
    length_score = len(claim.split()) / 10
    verb_score = any(token.pos_ == "VERB" for token in doc)

    return round(entity_score + length_score + (1 if verb_score else 0), 2)

In [25]:
def extract_claims_pipeline(text):

    # Step 1: LLM extraction
    claims = llm_extract_claims(text)

    # Step 2: Cleaning
    claims = clean_claims(claims)

    # Step 3: Remove hallucinations
    claims = remove_hallucinations(claims)

    # Step 4: Strict validation
    claims = strict_validation(claims, text)

    # Step 5: Scoring
    scored = [(c, score_claim(c)) for c in claims]

    return scored

In [26]:
results = extract_claims_pipeline(text)

for claim, score in results:
    print(f"{claim} --> Score: {score}")

Amazon was founded by Jeff Bezos. --> Score: 3.6
Amazon was founded in 1994. --> Score: 3.5
The company is headquartered in Washington. --> Score: 2.6
The company is headquartered in Seattle. --> Score: 2.6
